# PD Model Validation Example

This notebook demonstrates a compact end-to-end validation workflow for a synthetic Probability of Default (PD) model.

The objective is not to build the best possible machine learning model. The objective is to show how `valmetrics` can be used to assess model discrimination, calibration, stability, and portfolio concentration across development, validation, and out-of-time samples.

## Workflow

1. Generate synthetic credit-risk data.
2. Train a simple logistic regression model on the development sample.
3. Score development, validation, and out-of-time samples.
4. Construct rating grades from development-sample score cutoffs.
5. Calculate validation metrics using `valmetrics`.
6. Summarize key validation findings.

## Table of Contents

1. [Data generation and sample overview](#1-data-generation-and-sample-overview)
2. [Model training and scoring](#2-model-training-and-scoring)
3. [Rating grade construction](#3-rating-grade-construction)
4. [Discrimination analysis](#4-discrimination-analysis)

   - AUC-ROC;
   - Standard Gini;
   - Conservative Gini;
   - KS statistic.

5. [Calibration analysis](#5-calibration-analysis)

   - Hosmer-Lemeshow test;
   - Binomial calibration test;
   - Grouped calibration by rating grade.

6. [Stability analysis](#6-stability-analysis)

   - Continuous PSI for model scores and risk drivers;
   - Categorical PSI for rating grades.

7. [Concentration diagnostics](#7-concentration-diagnostics)

   - HHI;
   - HCI.

8. [Final validation summary](#8-final-validation-summary)


In [1]:
import numpy as np
import pandas as pd

## 1. Data generation and sample overview

In [2]:
import data_generator

df = data_generator.generate_pd_validation_data()
df.shape

(24000, 15)

In [3]:
df.groupby("sample").agg(
    observations=("default_12m", "size"),
    default_rate=("default_12m", "mean"),
    avg_dti=("debt_to_income", "mean"),
    avg_utilization=("credit_utilization", "mean"),
    avg_bureau_score=("bureau_score", "mean"),
)

,observations,default_rate,avg_dti,avg_utilization,avg_bureau_score
sample,,,,,
development,12000,0.076750,0.329760,0.426473,652.457731
out_of_time,6000,0.225000,0.423523,0.515966,599.844313
validation,6000,0.102333,0.357928,0.446269,635.589731


**Interpretation.**

The synthetic data shows a controlled deterioration from the development sample to the out-of-time sample. The out-of-time sample has a materially higher default rate, higher average debt-to-income ratio, higher credit utilization, and lower average bureau score.

This creates a useful validation setup: the model is trained on a relatively stable development population and then tested on both a regular validation sample and a shifted out-of-time population.

## 2. Model training and scoring

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [5]:
target_col = "default_12m"
sample_col = "sample"

numeric_features = [
    "age",
    "annual_income",
    "debt_to_income",
    "credit_utilization",
    "months_on_book",
    "delinquencies_12m",
    "bureau_score",
    "loan_to_value",
    "ead",
]

categorical_features = [
    "segment",
    "region",
]

train_df = df[df[sample_col] == "development"].copy()
validation_df = df[df[sample_col] == "validation"].copy()
oot_df = df[df[sample_col] == "out_of_time"].copy()

X_train = train_df[numeric_features + categorical_features]
y_train = train_df[target_col]

X_validation = validation_df[numeric_features + categorical_features]
y_validation = validation_df[target_col]

X_oot = oot_df[numeric_features + categorical_features]
y_oot = oot_df[target_col]

In [6]:
numeric_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("one_hot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_preprocessor, numeric_features),
        ("categorical", categorical_preprocessor, categorical_features),
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=1_000,
                solver="lbfgs",
                class_weight=None,
                random_state=42,
            ),
        ),
    ]
)

model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the differen

In [7]:
predicted_col = "y_prob"

train_df[predicted_col] = model.predict_proba(X_train)[:, 1]
validation_df[predicted_col] = model.predict_proba(X_validation)[:, 1]
oot_df[predicted_col] = model.predict_proba(X_oot)[:, 1]

**Interpretation.**

The logistic regression model is trained only on the development sample. Validation and out-of-time observations are not used for fitting, which avoids leakage.


## 3. Rating grade construction

In [8]:
rating_col = "rating_grade"

grade_labels = ["A", "B", "C", "D", "E", "F", "G"]
_, rating_edges = pd.qcut(
    train_df[predicted_col],
    q=len(grade_labels),
    retbins=True,
    duplicates="drop",
)


actual_n_grades = len(rating_edges) - 1
actual_grade_labels = grade_labels[:actual_n_grades]
rating_edges[0] = -np.inf
rating_edges[-1] = np.inf

for sample_df in [train_df, validation_df, oot_df]:
    sample_df[rating_col] = pd.cut(
        sample_df[predicted_col],
        bins=rating_edges,
        labels=actual_grade_labels,
        include_lowest=True,
    ).astype("object")

In [9]:
scored_df = pd.concat(
    [train_df, validation_df, oot_df],
    ignore_index=True,
)
dev_flag = scored_df["sample"] == "development"
val_flag = scored_df["sample"] == "validation"
oot_flag = scored_df["sample"] == "out_of_time"

**Interpretation.**

Rating grade cutoffs are derived from the development sample and then applied unchanged to validation and out-of-time samples. This avoids leakage from future samples into the rating scale construction.

The development sample is expected to be approximately evenly distributed across grades. A shift of validation or out-of-time observations toward worse grades indicates score distribution drift and can later be measured using categorical PSI and concentration diagnostics.

## 4. Discrimination analysis

In [10]:
from valmetrics.discrimination import (
    gini_conservative,
    gini_standard,
    ks_statistic,
    roc_auc,
)

In [11]:
def discrimination_summary(data: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for sample_name, sample_data in data.groupby("sample"):
        y_true = sample_data[target_col]
        y_score = sample_data[predicted_col]
        rows.append(
            {
                "sample": sample_name,
                "roc_auc": roc_auc(y_true, y_score),
                "gini": gini_standard(y_true, y_score),
                "gini_conservative": gini_conservative(y_true, y_score),
                "ks": ks_statistic(y_true, y_score),
                "default_rate": y_true.mean(),
                "mean_pd": y_score.mean(),
                "n_observations": len(sample_data),
            }
        )

    return pd.DataFrame(rows)


discrimination_summary(scored_df)

,sample,roc_auc,gini,gini_conservative,ks,default_rate,mean_pd,n_observations
0,development,0.732328,0.464655,0.464655,0.355726,0.076750,0.076757,12000
1,out_of_time,0.709977,0.419953,0.419953,0.313047,0.225000,0.143658,6000
2,validation,0.726177,0.452353,0.452353,0.344331,0.102333,0.091616,6000


**Interpretation.**

The model keeps positive discrimination power across all samples. AUC-ROC and Gini are highest on the development sample, slightly lower on the validation sample, and lower again on the out-of-time sample.

This indicates that the model still ranks higher-risk borrowers above lower-risk borrowers, but ranking power deteriorates under out-of-time population shift. The KS statistic follows the same pattern.

The gap between observed default rate and mean predicted probability is especially visible in the out-of-time sample. This is not a discrimination issue by itself; it points to calibration deterioration and should be reviewed in the calibration section.

## 5. Calibration analysis

In [12]:
from valmetrics.calibration import (
    binomial_test,
    grouped_binomial_test,
    grouped_hosmer_lemeshow,
    hosmer_lemeshow,
)

### Hosmer-Lemeshow test

In [26]:
hl_val = hosmer_lemeshow(
    scored_df[val_flag][target_col],
    scored_df[val_flag][predicted_col],
)
hl_oot = hosmer_lemeshow(
    scored_df[oot_flag][target_col],
    scored_df[oot_flag][predicted_col],
)

hosmer_lemeshow_summary = pd.DataFrame(
    [
        {
            "sample": "validation",
            "hl_statistic": hl_val.statistic,
            "hl_p_value": hl_val.p_value,
        },
        {
            "sample": "out_of_time",
            "hl_statistic": hl_oot.statistic,
            "hl_p_value": hl_oot.p_value,
        },
    ]
)

hosmer_lemeshow_summary

,sample,hl_statistic,hl_p_value
0,validation,17.424861,0.025977
1,out_of_time,391.231405,0.000000


**Interpretation.**

The Hosmer-Lemeshow test checks whether observed and expected defaults are aligned across probability groups.

A low p-value indicates evidence of group-level calibration deviation. The validation sample should be interpreted as regular model validation, while the out-of-time sample should be interpreted as a stress case under population drift.

The out-of-time result provides strong evidence that predicted probabilities underestimate observed default frequency across probability groups.


### Hosmer-Lemeshow test for rating grades

In [27]:
grp_hl_val = grouped_hosmer_lemeshow(
    scored_df[val_flag][target_col],
    scored_df[val_flag][predicted_col],
    scored_df[val_flag][rating_col],
)
grp_hl_oot = grouped_hosmer_lemeshow(
    scored_df[oot_flag][target_col],
    scored_df[oot_flag][predicted_col],
    scored_df[oot_flag][rating_col],
)

grp_hosmer_lemeshow_summary = pd.DataFrame(
    [
        {
            "sample": "validation",
            "hl_statistic": grp_hl_val.statistic,
            "hl_p_value": grp_hl_val.p_value,
        },
        {
            "sample": "out_of_time",
            "hl_statistic": grp_hl_oot.statistic,
            "hl_p_value": grp_hl_oot.p_value,
        },
    ]
)

grp_hosmer_lemeshow_summary

,sample,hl_statistic,hl_p_value
0,validation,12.718702,0.026162
1,out_of_time,374.808199,0.000000


**Interpretation.**

The grouped Hosmer-Lemeshow test evaluates calibration across rating grades rather than automatically constructed probability groups.

This is closer to a business validation view, because rating grades are often used for reporting, monitoring, and risk decisioning. A low p-value indicates that observed and expected defaults are not well aligned across the deployed grade structure.

If the grouped result is weaker than the automatic-group result, the issue is visible directly in the rating scale and should be investigated by grade-level observed versus expected default rates.


### Binomial test for PD deciles

In [15]:
binom_test_res = binomial_test(scored_df[oot_flag][target_col], scored_df[oot_flag][predicted_col])
df_binom_test_res = pd.DataFrame(binom_test_res.bins)
display(df_binom_test_res)
print("Alternative:", binom_test_res.alternative)
print("Confidence Level:", binom_test_res.confidence_level)
print("Method:", binom_test_res.method)

,group,n_observations,observed_defaults,expected_defaults,observed_dr,average_pd,lower_default_bound,upper_default_bound,lower_default_rate,upper_default_rate,p_value
0,0,600,42,17.434056,0.070000,0.029057,10,26,0.016667,0.043333,2.817163e-07
1,1,600,59,28.288906,0.098333,0.047148,19,39,0.031667,0.065000,1.927654e-07
2,2,600,74,37.693403,0.123333,0.062822,26,50,0.043333,0.083333,4.549424e-08
3,3,600,91,48.013424,0.151667,0.080022,35,61,0.058333,0.101667,6.656561e-09
4,4,600,95,59.000995,0.158333,0.098335,45,74,0.075000,0.123333,3.878117e-06
5,5,600,126,73.078461,0.210000,0.121797,58,89,0.096667,0.148333,1.002607e-09
6,6,600,142,90.695030,0.236667,0.151158,74,108,0.123333,0.180000,3.680688e-08
7,7,600,194,113.260141,0.323333,0.188767,95,132,0.158333,0.220000,3.739995e-15
8,8,600,206,151.006494,0.343333,0.251677,130,172,0.216667,0.286667,5.592072e-07
9,9,600,321,243.479348,0.535000,0.405799,220,267,0.366667,0.445000,2.097816e-10


Alternative: two-sided
Confidence Level: 0.95
Method: binomial_exact


**Interpretation.**

The binomial test compares observed defaults with expected defaults within each automatically constructed probability group.

Groups where observed defaults fall outside the reported default-count bounds or have low p-values should be treated as potential calibration exceptions. Since the test is performed group by group, it is useful for locating where calibration deterioration is concentrated.

The test uses the average PD within each group. Therefore, it should be interpreted as an exact binomial test under a group-level average-PD approximation, not as a full Poisson-binomial test using all individual PDs.


### Binomial test by rating grades

In [16]:
grp_binom_test_res = grouped_binomial_test(
    scored_df[oot_flag][target_col],
    scored_df[oot_flag][predicted_col],
    scored_df[oot_flag][rating_col],
)
df_grp_binom_test_res = pd.DataFrame(grp_binom_test_res.bins)
display(df_grp_binom_test_res)
print("Alternative:", grp_binom_test_res.alternative)
print("Confidence Level:", grp_binom_test_res.confidence_level)
print("Method:", grp_binom_test_res.method)

,group,n_observations,observed_defaults,expected_defaults,observed_dr,average_pd,lower_default_bound,upper_default_bound,lower_default_rate,upper_default_rate,p_value
0,D,632,72,35.137037,0.113924,0.055597,24,47,0.037975,0.074367,1.525151e-08
1,G,2443,870,604.268568,0.356120,0.247347,563,646,0.230454,0.264429,5.805479e-33
2,F,1153,214,125.898225,0.185603,0.109192,106,147,0.091934,0.127493,1.705397e-14
3,E,860,125,65.616698,0.145349,0.076298,51,81,0.059302,0.094186,7.967081e-12
4,C,462,40,19.167425,0.086580,0.041488,11,28,0.023810,0.060606,1.803146e-05
5,B,305,20,9.058618,0.065574,0.029700,4,15,0.013115,0.049180,1.036389e-03
6,A,145,9,2.803685,0.062069,0.019336,0,6,0.000000,0.041379,2.173597e-03


Alternative: two-sided
Confidence Level: 0.95
Method: binomial_exact


**Interpretation.**

The grouped binomial test shows calibration performance at the rating-grade level.

In the out-of-time sample, observed defaults are materially higher than expected defaults across multiple rating grades. Several groups have very low p-values, which indicates systematic risk underestimation rather than an isolated random deviation.

This is a strong validation finding: the model still ranks borrowers reasonably well, but the PD level is too low under the out-of-time population mix.


## 6. Stability analysis

In [17]:
from valmetrics.stability import psi_categorical, psi_continuous

### PSI by score

In [18]:
psi_score = psi_continuous(
    scored_df[dev_flag][predicted_col],
    scored_df[oot_flag][predicted_col],
)

pd.DataFrame(psi_score.bins)
display(pd.DataFrame(psi_score.bins))
psi_score.value

,lower_bound,upper_bound,expected_count,actual_count,expected_proportion,actual_proportion,contribution
0,-inf,0.020430,1200,76,0.1,0.012667,0.180448
1,0.020430,0.028073,1200,169,0.1,0.028167,0.091015
2,0.028073,0.035757,1200,235,0.1,0.039167,0.057022
3,0.035757,0.044540,1200,308,0.1,0.051333,0.032452
4,0.044540,0.054660,1200,403,0.1,0.067167,0.013067
5,0.054660,0.067788,1200,501,0.1,0.083500,0.002975
6,0.067788,0.085814,1200,606,0.1,0.101000,0.000010
7,0.085814,0.111051,1200,755,0.1,0.125833,0.005936
8,0.111051,0.161536,1200,1065,0.1,0.177500,0.044470
9,0.161536,inf,1200,1882,0.1,0.313667,0.244255


0.6716508058948963

**Interpretation.**

The PSI for model scores indicates a material distribution shift between the development and out-of-time samples.

The out-of-time sample has a much larger share of observations in high-risk score bins and a much smaller share in low-risk score bins. This confirms that the model is being applied to a population that is materially different from the development population.

This supports the calibration finding: model deterioration is likely driven not only by random noise, but by a real population shift.


### PSI by rating grades

In [19]:
psi_rating = psi_categorical(
    scored_df[dev_flag][rating_col],
    scored_df[oot_flag][rating_col],
)

pd.DataFrame(psi_rating.bins)
display(pd.DataFrame(psi_rating.bins))
psi_rating.value

,category,expected_count,actual_count,expected_proportion,actual_proportion,contribution
0,F,1714,1153,0.142833,0.192167,0.014636
1,E,1714,860,0.142833,0.143333,0.000002
2,C,1714,462,0.142833,0.077000,0.040677
3,D,1714,632,0.142833,0.105333,0.011421
4,G,1715,2443,0.142917,0.407167,0.276659
5,B,1714,305,0.142833,0.050833,0.095048
6,A,1715,145,0.142917,0.024167,0.211053


0.649495302159685

**Interpretation.**

The categorical PSI by rating grade confirms a material shift in the rating distribution.

The out-of-time sample is concentrated much more heavily in the worst rating grade and less represented in the best grades. This means the portfolio mix has migrated toward higher-risk borrowers.

This is important for validation because grade-level monitoring, calibration tests, and concentration diagnostics are all affected by this shift.


### PSI by categorical features

In [20]:
for feature in categorical_features:
    psi = psi_categorical(
        scored_df[dev_flag][feature],
        scored_df[oot_flag][feature],
        missing="separate",
    )
    print(feature, psi.value)

segment 0.050850793330312616
region 0.0019021508813719212


In [21]:
psi_segment = psi_categorical(scored_df[dev_flag]["segment"], scored_df[oot_flag]["segment"])
display(pd.DataFrame(psi_segment.bins))

,category,expected_count,actual_count,expected_proportion,actual_proportion,contribution
0,unsecured,3855,2354,0.321250,0.392333,0.014209
1,mortgage,6071,2364,0.505917,0.394000,0.027982
2,sme,2074,1282,0.172833,0.213667,0.008660


**Interpretation.**

Categorical PSI shows limited shift in region and a mild shift in segment composition.

The segment shift is visible but not the main source of instability. Region distribution is broadly stable. Therefore, the main out-of-time drift is more likely driven by risk-related numeric variables and score distribution rather than by regional portfolio mix.


### PSI by numeric features

In [22]:
for feature in numeric_features:
    psi = psi_continuous(
        scored_df[dev_flag][feature],
        scored_df[oot_flag][feature],
        missing="separate",
    )
    print(feature, psi.value)

age 0.007043917689461752
annual_income 0.03581714704368011
debt_to_income 0.4113973727827937
credit_utilization 0.19239033814491102
months_on_book 0.00411300213107708
delinquencies_12m 0.04900580543465288
bureau_score 0.9326664375722021
loan_to_value 0.0580001311695957
ead 0.030398257111668367


In [23]:
psi_bureau = psi_continuous(
    scored_df[dev_flag]["bureau_score"],
    scored_df[oot_flag]["bureau_score"],
    missing="separate",
)
display(pd.DataFrame(psi_bureau.bins))
print(psi_bureau.value)

,lower_bound,upper_bound,expected_count,actual_count,expected_proportion,actual_proportion,contribution
0,-inf,591.0,1184,2444,0.098667,0.407333,0.437654
1,591.0,612.0,1142,871,0.095167,0.145167,0.021113
2,612.0,628.0,1225,643,0.102083,0.107167,0.000247
3,628.0,641.0,1135,421,0.094583,0.070167,0.007291
4,641.0,654.0,1257,414,0.104750,0.069000,0.014925
5,654.0,666.0,1184,275,0.098667,0.045833,0.040509
6,666.0,679.0,1178,245,0.098167,0.040833,0.050291
7,679.0,693.0,1161,186,0.096750,0.031000,0.074833
8,693.0,713.1,1244,148,0.103667,0.024667,0.113422
9,713.1,inf,1190,121,0.099167,0.020167,0.125829


0.9326664375722021


**Interpretation.**

Numeric PSI identifies the main drivers of population shift.

The largest shift is observed in bureau score, followed by debt-to-income and credit utilization. These are economically meaningful credit-risk drivers, so the drift is directly relevant for model performance.

The out-of-time population is not just statistically different; it is riskier along variables that are expected to affect default probability.


The detailed bureau score PSI table shows that the out-of-time sample has a much higher concentration in the lowest bureau-score bin and a much lower concentration in high-score bins.

This is a clear adverse credit-quality shift. It helps explain both the higher observed default rate and the deterioration in model calibration.


## 7. Concentration diagnostics

In [24]:
from valmetrics.diagnostics import hci, herfindahl_hirschman

In [25]:
diagnostics_summary = pd.DataFrame(
    [
        {
            "grouping": "rating_grade",
            "hhi": herfindahl_hirschman(scored_df[oot_flag][rating_col]),
            "adjusted_hhi": herfindahl_hirschman(
                scored_df[oot_flag][rating_col],
                normalized=True,
                n_groups=7,
            ),
            "hci": hci(scored_df[oot_flag][rating_col]).value,
            "hci_groups": hci(scored_df[oot_flag][rating_col]).groups,
        },
        {
            "grouping": "segment",
            "hhi": herfindahl_hirschman(scored_df[oot_flag]["segment"]),
            "adjusted_hhi": herfindahl_hirschman(
                scored_df[oot_flag]["segment"],
                normalized=True,
                n_groups=3,
            ),
            "hci": hci(scored_df[oot_flag]["segment"]).value,
            "hci_groups": hci(scored_df[oot_flag]["segment"]).groups,
        },
    ]
)

diagnostics_summary

,grouping,hhi,adjusted_hhi,hci,hci_groups
0,rating_grade,0.243449,0.117358,0.407167,"(G,)"
1,segment,0.354815,0.032222,0.394000,"(mortgage,)"


**Interpretation.**

Concentration diagnostics show that the out-of-time rating distribution is meaningfully concentrated in the worst rating grade. The largest rating grade contains more than 40% of observations.

This concentration matters for validation because portfolio conclusions can become dominated by one grade. It also confirms the PSI result: the out-of-time sample has shifted toward high-risk ratings.

Segment concentration is less concerning. The largest segment is mortgage, but the adjusted HHI is low, indicating that segment distribution is not the main validation issue.



## 8. Final validation summary


The model shows acceptable but deteriorating discrimination performance. AUC-ROC, Gini, and KS remain positive across development, validation, and out-of-time samples, which means the model still ranks borrowers by risk. However, discrimination weakens in the out-of-time sample.

The main issue is calibration. The out-of-time sample has a much higher observed default rate than the model-implied mean PD. Group-level calibration tests indicate that observed defaults exceed expected defaults across multiple probability groups and rating grades. This suggests systematic underestimation of risk under the out-of-time population.

Stability metrics confirm material population drift. PSI is high for the model score and rating grade distribution. Numeric PSI identifies bureau score, debt-to-income, and credit utilization as the main drivers of the shift. These are economically meaningful risk drivers, so the drift is relevant for model performance.

Concentration diagnostics show that the out-of-time portfolio is heavily concentrated in the worst rating grade. This concentration reinforces the validation concern because the portfolio mix has moved toward materially riskier borrowers.

Overall conclusion: the model keeps useful ranking power, but its probability calibration is not stable under out-of-time population shift.
